# ResNet50 Training Refinements

This notebook starts from the evaluated ResNet50 baseline and tests a
more disciplined fine-tuning recipe: stronger augmentation, label
smoothing, AdamW, learning-rate scheduling, and early stopping. It is
intentionally separate from the baseline notebook so improvements can
be attributed to training changes rather than evaluation changes.

## 1. Experiment Plan

| Experiment | Change | Purpose |
| --- | --- | --- |
| FT-V2 | longer fine-tuning + early stopping | test whether the baseline is undertrained |
| FT-V3 | scheduler + AdamW | stabilize deeper fine-tuning |
| FT-V4 | stronger augmentation + label smoothing | improve robustness to plating, lighting, and label noise |

The final evaluation should reuse the same top-1, top-5, class report,
confusion-pair, and qualitative-error protocol from notebook 1.

In [ ]:
import random
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

In [ ]:
@dataclass(frozen=True)
class CFG:
    """Configuration for ResNet50 refinement experiments."""

    MODE: str = "train"  # Options: "train" or "evaluate"
    SEED: int = 42
    BATCH_SIZE: int = 32
    IMAGE_SIZE: tuple[int, int] = (224, 224)
    NUM_CLASSES: int = 101
    NUM_WORKERS: int = 2
    MAX_EPOCHS: int = 15
    PATIENCE: int = 3
    LEARNING_RATE: float = 3e-5
    WEIGHT_DECAY: float = 1e-4
    LABEL_SMOOTHING: float = 0.05
    TOP_K: int = 5
    HARD_CLASS_COUNT: int = 12
    ERROR_EXAMPLE_COUNT: int = 12
    LATENCY_WARMUP_RUNS: int = 10
    LATENCY_BENCHMARK_RUNS: int = 50
    TRAINABLE_PREFIXES: tuple[str, ...] = ("layer3", "layer4", "fc")
    DATA_DIR: Path = Path("/kaggle/input/datasets/kmader/food41")
    BASELINE_ARTIFACT_DIR: Path = Path(
        "/kaggle/input/models/tuannm3823/food101-baseline-artifacts/"
        "pytorch/default/1/food101-baseline-artifacts"
    )
    REFINED_ARTIFACT_DIR: Path = Path(
        "/kaggle/input/models/tuannm3823/food101-resnet50-refinements/"
        "pytorch/default/1/food101-resnet50-refinements"
    )
    RESULTS_DIR: Path = Path("/kaggle/working/results/resnet50_refinements")
    FIGURES_DIR: Path = Path("/kaggle/working/results/resnet50_refinements/figures")
    BASELINE_CHECKPOINT_NAME: str = "finetuned_exp_2_layer3_4.pth"
    REFINED_CHECKPOINT_NAME: str = "resnet50_ft_v2_best.pth"


assert CFG.MODE in {"train", "evaluate"}, "CFG.MODE must be 'train' or 'evaluate'."
CFG.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CFG.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Mode: {CFG.MODE}")

In [ ]:
def create_data_manifest(image_dir: Path) -> pd.DataFrame:
    """Create an image-path and label manifest from class folders."""
    records = []
    for class_dir in sorted(path for path in image_dir.iterdir() if path.is_dir()):
        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                records.append({"path": str(image_path), "label": class_dir.name})
    return pd.DataFrame.from_records(records)


IMAGE_DIR = CFG.DATA_DIR / "images"
df = create_data_manifest(IMAGE_DIR)
class_names = sorted(df["label"].unique())
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=CFG.SEED,
    stratify=df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=CFG.SEED,
    stratify=temp_df["label"],
)
print(len(train_df), len(val_df), len(test_df))

In [ ]:
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

TRAIN_TRANSFORMS = transforms.Compose(
    [
        transforms.RandomResizedCrop(CFG.IMAGE_SIZE, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply(
            [transforms.ColorJitter(0.2, 0.2, 0.2, 0.05)],
            p=0.6,
        ),
        transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ]
)

VAL_TRANSFORMS = transforms.Compose(
    [
        transforms.Resize(CFG.IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ]
)

In [ ]:
class FoodDataset(Dataset):
    """Food-101 dataset wrapper."""

    def __init__(self, dataframe, class_to_idx, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, self.class_to_idx[row["label"]]


train_loader = DataLoader(
    FoodDataset(train_df, class_to_idx, TRAIN_TRANSFORMS),
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=device.type == "cuda",
)
val_loader = DataLoader(
    FoodDataset(val_df, class_to_idx, VAL_TRANSFORMS),
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=device.type == "cuda",
)
test_loader = DataLoader(
    FoodDataset(test_df, class_to_idx, VAL_TRANSFORMS),
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=device.type == "cuda",
)

In [ ]:
def make_classifier_head(in_features: int) -> nn.Sequential:
    """Create the Food-101 classifier head."""
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, CFG.NUM_CLASSES),
    )


def build_resnet50() -> nn.Module:
    """Create ResNet50 with the project classifier head."""
    model = models.resnet50(weights=None)
    model.fc = make_classifier_head(model.fc.in_features)
    return model


def resolve_checkpoint(filename: str, search_roots: list[Path]) -> Path:
    """Find a checkpoint in known Kaggle working or artifact roots."""
    candidates = []
    for root in search_roots:
        candidates.extend([root / filename, root / "results" / filename])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    for root in search_roots:
        if root.exists():
            matches = sorted(root.rglob(filename))
            if matches:
                return matches[0]

    searched = "\n".join(str(root) for root in search_roots)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")


def resolve_baseline_checkpoint() -> Path:
    """Find the baseline checkpoint from attached Kaggle artifacts."""
    return resolve_checkpoint(
        CFG.BASELINE_CHECKPOINT_NAME,
        [CFG.BASELINE_ARTIFACT_DIR, CFG.RESULTS_DIR],
    )


def resolve_refined_checkpoint() -> Path:
    """Find the best refined ResNet50 checkpoint."""
    return resolve_checkpoint(
        CFG.REFINED_CHECKPOINT_NAME,
        [CFG.RESULTS_DIR, CFG.REFINED_ARTIFACT_DIR],
    )


def configure_trainable_layers(model: nn.Module) -> None:
    """Freeze all layers except selected fine-tuning prefixes."""
    for name, parameter in model.named_parameters():
        parameter.requires_grad = name.startswith(CFG.TRAINABLE_PREFIXES)

In [ ]:
model = build_resnet50()
if CFG.MODE == "train":
    checkpoint = resolve_baseline_checkpoint()
else:
    checkpoint = resolve_refined_checkpoint()

model.load_state_dict(torch.load(checkpoint, map_location=device))
configure_trainable_layers(model)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)
optimizer = optim.AdamW(
    (parameter for parameter in model.parameters() if parameter.requires_grad),
    lr=CFG.LEARNING_RATE,
    weight_decay=CFG.WEIGHT_DECAY,
)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
print(f"Loaded checkpoint: {checkpoint}")

In [ ]:
def run_epoch(model, loader, optimizer=None):
    """Run one train or evaluation epoch."""
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            correct += outputs.argmax(dim=1).eq(labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), 100.0 * correct / total


if CFG.MODE == "train":
    best_acc = -np.inf
    patience_count = 0
    history = []
    best_path = CFG.RESULTS_DIR / CFG.REFINED_CHECKPOINT_NAME

    for epoch in range(1, CFG.MAX_EPOCHS + 1):
        start_time = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader)
        scheduler.step(val_acc)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "lr": optimizer.param_groups[0]["lr"],
            }
        )
        print(
            f"Epoch {epoch}: val_acc={val_acc:.2f}% "
            f"train_acc={train_acc:.2f}% time={time.time() - start_time:.1f}s"
        )

        if val_acc > best_acc:
            best_acc = val_acc
            patience_count = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_count += 1
            if patience_count >= CFG.PATIENCE:
                print("Early stopping triggered.")
                break

    pd.DataFrame(history).to_csv(
        CFG.RESULTS_DIR / "resnet50_ft_v2_history.csv",
        index=False,
    )
    print(f"Best validation accuracy: {best_acc:.2f}%")
else:
    print("Training skipped in evaluate mode.")

## 2. Final Evaluation

This section mirrors the baseline notebook's evaluation contract so the refined
ResNet50 can be compared directly against the original fine-tuned checkpoint.
It reports validation/test top-1 and top-5 accuracy, per-class F1, confusion
pairs, high-confidence errors, and inference efficiency.

In [ ]:
def load_best_refined_model() -> nn.Module:
    """Load the best refined checkpoint for evaluation."""
    checkpoint = resolve_refined_checkpoint()
    eval_model = build_resnet50()
    eval_model.load_state_dict(torch.load(checkpoint, map_location=device))
    eval_model = eval_model.to(device)
    eval_model.eval()
    print(f"Evaluating checkpoint: {checkpoint}")
    return eval_model


def collect_classification_outputs(model, loader, top_k: int = CFG.TOP_K):
    """Collect prediction rows and top-k metrics."""
    model.eval()
    rows = []
    top_1_correct = 0
    top_k_correct = 0
    total = 0
    manifest = loader.dataset.df

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probabilities = F.softmax(outputs, dim=1)
            top_probabilities, top_indices = torch.topk(probabilities, top_k, dim=1)
            predictions = top_indices[:, 0].cpu()
            labels_cpu = labels.cpu()

            top_1_correct += predictions.eq(labels_cpu).sum().item()
            top_k_correct += top_indices.cpu().eq(labels_cpu.unsqueeze(1)).any(dim=1).sum().item()
            batch_start = total
            total += labels_cpu.size(0)

            for row_idx, true_idx in enumerate(labels_cpu.tolist()):
                pred_idx = predictions[row_idx].item()
                manifest_row = manifest.iloc[batch_start + row_idx]
                rows.append(
                    {
                        "path": manifest_row["path"],
                        "true_label": class_names[true_idx],
                        "pred_label": class_names[pred_idx],
                        "confidence": top_probabilities[row_idx, 0].item(),
                        "is_correct": pred_idx == true_idx,
                    }
                )

    metrics_df = pd.DataFrame(
        [
            {"metric": "top_1_accuracy", "value": 100.0 * top_1_correct / total},
            {"metric": f"top_{top_k}_accuracy", "value": 100.0 * top_k_correct / total},
        ]
    )
    return pd.DataFrame(rows), metrics_df


def classification_report_frame(prediction_df: pd.DataFrame) -> pd.DataFrame:
    """Create per-class precision, recall, and F1 report."""
    report = classification_report(
        prediction_df["true_label"],
        prediction_df["pred_label"],
        labels=class_names,
        output_dict=True,
        zero_division=0,
    )
    return pd.DataFrame(report).transpose().loc[class_names].sort_values(
        "f1-score",
        ascending=False,
    )


def confusion_pair_frame(prediction_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize repeated true-label to predicted-label errors."""
    errors = prediction_df[~prediction_df["is_correct"]]
    return (
        errors.groupby(["true_label", "pred_label"], as_index=False)
        .agg(count=("path", "size"), mean_confidence=("confidence", "mean"))
        .sort_values(["count", "mean_confidence"], ascending=False)
        .reset_index(drop=True)
    )


def evaluate_split(model, loader, split_name: str):
    """Evaluate one split and export all tabular diagnostics."""
    prediction_df, metrics_df = collect_classification_outputs(model, loader)
    report_df = classification_report_frame(prediction_df)
    confusion_pairs_df = confusion_pair_frame(prediction_df)

    prediction_df.to_csv(CFG.RESULTS_DIR / f"{split_name}_predictions.csv", index=False)
    metrics_df.to_csv(CFG.RESULTS_DIR / f"{split_name}_metrics.csv", index=False)
    report_df.to_csv(CFG.RESULTS_DIR / f"{split_name}_class_report.csv")
    confusion_pairs_df.to_csv(
        CFG.RESULTS_DIR / f"{split_name}_confusion_pairs.csv",
        index=False,
    )

    print(f"{split_name.title()} metrics")
    display(metrics_df)
    print(f"{split_name.title()} hardest classes")
    display(report_df.tail(10)[["precision", "recall", "f1-score", "support"]])
    print(f"{split_name.title()} top confusion pairs")
    display(confusion_pairs_df.head(10))
    return prediction_df, metrics_df, report_df, confusion_pairs_df

In [ ]:
def model_efficiency_summary(model: nn.Module) -> pd.DataFrame:
    """Measure parameter count, memory footprint, and single-image latency."""
    model.eval()
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    buffer_count = sum(buffer.numel() for buffer in model.buffers())
    model_size_mb = 4 * (parameter_count + buffer_count) / 1024**2
    sample = torch.randn(1, 3, *CFG.IMAGE_SIZE).to(device)

    with torch.no_grad():
        for _ in range(CFG.LATENCY_WARMUP_RUNS):
            _ = model(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
        start_time = time.perf_counter()
        for _ in range(CFG.LATENCY_BENCHMARK_RUNS):
            _ = model(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start_time

    return pd.DataFrame(
        [
            {
                "parameters": parameter_count,
                "model_size_mb": model_size_mb,
                "device": str(device),
                "latency_ms_per_image": 1000 * elapsed / CFG.LATENCY_BENCHMARK_RUNS,
            }
        ]
    )


refined_model = load_best_refined_model()
val_pred_df, val_metrics_df, val_report_df, val_confusion_pairs_df = evaluate_split(
    refined_model,
    val_loader,
    "val",
)
test_pred_df, test_metrics_df, test_report_df, test_confusion_pairs_df = evaluate_split(
    refined_model,
    test_loader,
    "test",
)

error_examples_df = (
    test_pred_df[~test_pred_df["is_correct"]]
    .sort_values("confidence", ascending=False)
    .head(CFG.ERROR_EXAMPLE_COUNT)
)
error_examples_df.to_csv(
    CFG.RESULTS_DIR / "qualitative_error_examples.csv",
    index=False,
)
efficiency_df = model_efficiency_summary(refined_model)
efficiency_df.to_csv(CFG.RESULTS_DIR / "model_efficiency.csv", index=False)

print("High-confidence error examples")
display(error_examples_df)
print("Efficiency")
display(efficiency_df)